# 21 — Frozen READ estimators and cross-fit profiles

This notebook computes only the seven frozen READ candidates. R2/R3
paths are fit on training folds; no held-out A attribution enters R2 profiles.
No science, controls, alpha sweep, tests, or lint are run.

In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import math
import torch

from src.model_utils import load_model, set_seed
from src.read_validation import (
    CANDIDATE_NAMES,
    READ_VALIDATION_PROTOCOL,
    READ_VALIDATION_PROTOCOL_SHA256,
    coordinate_resampling_edits_from_manifest,
    fit_all_fold_component_paths,
    path_localization_from_edits,
    read_json,
    r3_component_profiles,
    score_under_all_fold_paths,
    write_json,
)
from src.v2_recalibration import language_mass_metric

ROOT = Path.cwd()
RAW_DIR = ROOT / 'data/raw/v5'
implementation_sha256 = hashlib.sha256((ROOT / 'src/read_validation.py').read_bytes()).hexdigest()
raw20_path = RAW_DIR / '20_read_ground_truth.json'
raw20_sha256 = hashlib.sha256(raw20_path.read_bytes()).hexdigest()
metrics20 = read_json(ROOT / 'results/metrics.json')['read_validation_v5']
assert metrics20['stage20']['raw_artifact']['sha256'] == raw20_sha256
raw20 = read_json(raw20_path)
assert raw20['protocol_sha256'] == READ_VALIDATION_PROTOCOL_SHA256
assert raw20['implementation_sha256'] == implementation_sha256
ground_rows = raw20['rows']
assert len(ground_rows) == 163
assert len({row['row_id'] for row in ground_rows}) == 163

set_seed(READ_VALIDATION_PROTOCOL['seed'])
bundle = load_model(
    READ_VALIDATION_PROTOCOL['model']['id'],
    revision=READ_VALIDATION_PROTOCOL['model']['revision'],
    dtype=torch.bfloat16,
)
assert bundle.revision == READ_VALIDATION_PROTOCOL['model']['revision']
assert next(bundle.hf_model.parameters()).dtype == torch.bfloat16
device = next(bundle.hf_model.parameters()).device
direction_bank_artifact = raw20['direction_bank_cache']
direction_bank_path = ROOT / direction_bank_artifact['path']
assert direction_bank_path.stat().st_size == direction_bank_artifact['bytes']
assert hashlib.sha256(direction_bank_path.read_bytes()).hexdigest() == direction_bank_artifact['sha256']
bank_payload = torch.load(direction_bank_path, map_location='cpu', weights_only=False)
assert bank_payload['protocol_sha256'] == READ_VALIDATION_PROTOCOL_SHA256
bank = {
    int(token_id): {int(layer): vector.to(device) for layer, vector in layers.items()}
    for token_id, layers in bank_payload['bank'].items()
}

def metric_from_payload(payload):
    if payload['type'] == 'next_token_difference':
        target = int(payload['target_token_id']); foil = int(payload['foil_token_id'])
        return lambda logits: logits[0, -1, target].float() - logits[0, -1, foil].float()
    positions = list(payload['score_positions']); source = list(payload['source_token_ids']); english = list(payload['english_token_ids'])
    return lambda logits: language_mass_metric(logits, positions, source, english)

def canonical_sha256(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':')).encode()).hexdigest()

print({'model': bundle.model_id, 'revision': bundle.revision, 'rows': len(ground_rows), 'candidates': CANDIDATE_NAMES})

In [ ]:
localization_checkpoint = RAW_DIR / '21_path_localization_checkpoint.json'
localization_header = {
    'protocol_sha256': READ_VALIDATION_PROTOCOL_SHA256,
    'implementation_sha256': implementation_sha256,
    'upstream_raw20_sha256': raw20_sha256,
    'direction_bank_sha256': direction_bank_artifact['sha256'],
    'model_revision': bundle.revision,
    'model_dtype': str(next(bundle.hf_model.parameters()).dtype),
    'expected_row_ids': sorted(row['row_id'] for row in ground_rows),
}
localized = {}
if localization_checkpoint.exists():
    payload = read_json(localization_checkpoint)
    if all(payload.get(key) == value for key, value in localization_header.items()):
        localized = {row['row_id']: row for row in payload.get('rows', [])}

for index, base in enumerate(ground_rows, start=1):
    row_id = base['row_id']
    if row_id in localized:
        continue
    if base['ground_truth_A'].get('causal_abs') is None or base.get('A_coordinate_manifest') is None:
        path = {'status': 'UNDEFINED_GROUND_TRUTH_A', 'mlps': [], 'attention_heads': []}
    else:
        input_ids = torch.tensor([base['input_token_ids']], dtype=torch.long, device=device)
        directions = {layer: bank[int(base['source_token_id'])][layer] for layer in READ_VALIDATION_PROTOCOL['workspace_layers']}
        edits = coordinate_resampling_edits_from_manifest(directions, base['A_coordinate_manifest']['coefficients'])
        path = path_localization_from_edits(
            bundle.hf_model,
            bundle.lens_model.layers,
            input_ids,
            metric_from_payload(base['metric_payload']),
            edits,
        )
        expected_a = base['ground_truth_A']
        assert math.isclose(path['clean_metric'], expected_a['clean_metric'], rel_tol=0.0, abs_tol=1e-5)
        assert math.isclose(path['a_actual_delta'], expected_a['signed_delta_edited_minus_clean'], rel_tol=0.0, abs_tol=1e-5)
    localized[row_id] = {
        'row_id': row_id, 'concept_id': base['concept_id'], 'fold_group': base['fold_group'],
        'fold': base['fold'], 'label': base['label'], 'class_name': base['class_name'],
        'path_localization': path,
    }
    write_json(localization_checkpoint, {**localization_header, 'rows': [localized[key] for key in sorted(localized)]})
    print(f'path localization {index}/{len(ground_rows)} {row_id}: {path["status"]}')
    gc.collect(); torch.cuda.empty_cache()

path_rows = [localized[row['row_id']] for row in ground_rows]
fold_paths = fit_all_fold_component_paths(path_rows)
union_components = sorted({component for path in fold_paths.values() for component in path['R2_R3']['component_ids']})
fold_paths_sha256 = canonical_sha256(fold_paths)
localization_rows_sha256 = canonical_sha256(path_rows)
print(json.dumps({fold: {'status': path['status'], 'train_complete': path['training_path_coverage_complete'], 'R2_R3': path['R2_R3']['component_ids']} for fold, path in fold_paths.items()}, indent=2))
print('R3 union components', len(union_components), union_components)

In [ ]:
scoring_checkpoint = RAW_DIR / '21_read_scores_checkpoint.json'
scoring_header = {
    'protocol_sha256': READ_VALIDATION_PROTOCOL_SHA256,
    'implementation_sha256': implementation_sha256,
    'upstream_raw20_sha256': raw20_sha256,
    'direction_bank_sha256': direction_bank_artifact['sha256'],
    'localization_rows_sha256': localization_rows_sha256,
    'fold_paths_sha256': fold_paths_sha256,
    'union_components': union_components,
    'expected_row_ids': sorted(row['row_id'] for row in ground_rows),
}
scored = {}
if scoring_checkpoint.exists():
    payload = read_json(scoring_checkpoint)
    if all(payload.get(key) == value for key, value in scoring_header.items()):
        scored = {row['row_id']: row for row in payload.get('rows', [])}

localized_by_id = {row['row_id']: row['path_localization'] for row in path_rows}
weight_component_cache = {}
for index, base in enumerate(ground_rows, start=1):
    row_id = base['row_id']
    if row_id in scored:
        continue
    input_ids = torch.tensor([base['input_token_ids']], dtype=torch.long, device=device)
    directions = {layer: bank[int(base['source_token_id'])][layer] for layer in range(24, 28)}
    r3 = r3_component_profiles(
        bundle.hf_model,
        bundle.lens_model.layers,
        input_ids,
        directions,
        union_components,
    )
    support = score_under_all_fold_paths(
        bundle,
        directions,
        localized_by_id[row_id],
        fold_paths,
        r3,
        base['carrying_mask']['positions'],
        row_id=row_id,
        concept_id=base['concept_id'],
        sequence_length=int(base['sequence_length']),
        weight_component_cache=weight_component_cache,
    )
    scored[row_id] = {
        **base,
        'path_localization': localized_by_id[row_id],
        'R3_component_profiles': r3,
        'scores_by_fold_path': support['scores_by_fold_path'],
        'score_status': support['status'],
    }
    write_json(scoring_checkpoint, {**scoring_header, 'rows': [scored[key] for key in sorted(scored)]})
    assigned = support['scores_by_fold_path'][str(base['fold'])]['scores']
    print(f'READ scores {index}/{len(ground_rows)} {row_id}: {assigned}')
    gc.collect(); torch.cuda.empty_cache()

scored_rows = [scored[row['row_id']] for row in ground_rows]
assert len(scored_rows) == 163
assert all(set(row['scores_by_fold_path']) == {'0', '1', '2', '3'} for row in scored_rows)
assert all(
    set(fold_record['scores']) == set(CANDIDATE_NAMES)
    for row in scored_rows
    for fold_record in row['scores_by_fold_path'].values()
)
print('scoring complete', len(scored_rows))

In [ ]:
raw21 = {
    'schema_version': 'read-estimators-v1',
    'protocol_sha256': READ_VALIDATION_PROTOCOL_SHA256,
    'implementation_sha256': implementation_sha256,
    'upstream_raw20_sha256': raw20_sha256,
    'direction_bank_sha256': direction_bank_artifact['sha256'],
    'localization_rows_sha256': localization_rows_sha256,
    'fold_paths_sha256': fold_paths_sha256,
    'union_components': union_components,
    'candidate_names': list(CANDIDATE_NAMES),
    'fold_paths': fold_paths,
    'rows': scored_rows,
}
raw21_artifact = write_json(RAW_DIR / '21_read_estimators.json', raw21)

metrics_path = ROOT / 'results/metrics.json'
metrics = read_json(metrics_path)
v5 = metrics['read_validation_v5']
assert v5['protocol_sha256'] == READ_VALIDATION_PROTOCOL_SHA256
v5['stage21'] = {
    'status': 'COMPLETE',
    'raw_artifact': raw21_artifact,
    'upstream_raw20_sha256': raw20_sha256,
    'direction_bank_sha256': direction_bank_artifact['sha256'],
    'fold_paths_sha256': fold_paths_sha256,
    'candidate_names': list(CANDIDATE_NAMES),
    'fold_paths': {
        fold: {
            'training_path_coverage_complete': path['training_path_coverage_complete'],
            'unavailable_training_row_ids': path['unavailable_training_row_ids'],
            'R1_n_components': path['R1']['n_components'],
            'R2_R3_component_ids': path['R2_R3']['component_ids'],
        }
        for fold, path in fold_paths.items()
    },
    'rows': [
        {
            'row_id': row['row_id'], 'concept_id': row['concept_id'], 'fold': row['fold'],
            'label': row['label'], 'class_name': row['class_name'],
            'assigned_fold_scores': row['scores_by_fold_path'][str(row['fold'])]['scores'],
            'score_status': row['score_status'],
        }
        for row in scored_rows
    ],
}
v5['stage22'] = {'status': 'PENDING'}
write_json(metrics_path, metrics)

from src.model_utils import release_model
del bank, bank_payload
release_model(bundle)
print(json.dumps({'raw21': raw21_artifact, 'candidates': list(CANDIDATE_NAMES)}, indent=2))